### Classes

In [ ]:
class Dog:
    species = "Canine"  # Class attribute

    def __init__(self, name, age):
        self.name = name  # Instance attribute
        self.age = age  # Instance attribute
        
# Creating an object of the Dog class
dog = Dog("Buddy", 3)
print(f"Name: {dog.name}, Age: {dog.age}, Species: {dog.species}")

# Converting object attributes to JSON format
import json
print(json.dumps(dog.__dict__, indent=4))

**Note**: As demonstrated in the above class, we manually defined the constructor and attributes using self. This process can be simplified by utilizing the Pydantic BaseModel, as shown below.

### Using Pydantic Base Model

In [ ]:
import uuid
from datetime import datetime, timezone

In [ ]:
from pydantic import BaseModel, Field

class AdditionalDetails(BaseModel):
    email: str = Field(default=None, description="The email address of the user")
    age: int = Field(default=None, description="The age of the user")

class UserAddress(BaseModel):
    street: str = Field(default=None, description="The street address of the user")
    city: str = Field(default=None, description="The city of the user")
    state: str = Field(default=None, description="The state of the user")
    country: str = Field(default=None, description="The country of the user")
    pin_code: int = Field(default=None, description="The pin code of the user's address")

class UserDetails(BaseModel):
    id: str = Field(default_factory=lambda: str(uuid.uuid4()), description="The unique identifier for the user")
    name: str = Field(default=None, description="The full name of the user")
    phone_number: str = Field(default=None, description="The phone number of the user")
    additional_details: AdditionalDetails = Field(default_factory=AdditionalDetails, description="Additional details about the user")
    address: list[UserAddress] = Field(default_factory=list, description="List of address associated with the user")
    created_at: str = Field(default_factory=lambda: datetime.now(timezone.utc).isoformat(), description="The timestamp when the user was created")
    updated_at: str = Field(default_factory=lambda: datetime.now(timezone.utc).isoformat(), description="The timestamp when the user was last updated")

In [ ]:
user_details = UserDetails(name="John Doe", phone_number="+1234567890")
user_details.additional_details = AdditionalDetails(email="john.doe@example.com", age=30)
user_details.address.append(UserAddress(street="123 Main St", city="Anytown", state="Anystate", country="Anycountry", pin_code=123456))
user_details.address.append(UserAddress(street="456 Side St", city="Othertown", state="Otherstate", country="Othercountry", pin_code=654321))

# Spread the UserDetails model into the JSON dictionary.
# In Pydantic v2, BaseModel provides model_dump_json() to serialize a model to JSON. 
print(user_details.model_dump_json(indent=4))

In [ ]:
jane_user_details = {
    "id": str(uuid.uuid4()),
    "name": "Jane Smith",
    "phone_number": "+0987654321",
    "additional_details": {
        "email": "jane.smith@example.com",
        "age": 28
    },
    "address": [
        {
            "street": "789 Central Ave",
            "city": "Middletown",
            "state": "Middlestate",
            "country": "Middlecountry",
            "pin_code": 654321
        }
    ],
    "created_at": "2024-06-01T12:00:00.000000+00:00",
    "updated_at": "2024-06-01T12:30:00.000000+00:00",
}

# Spread the JSON dictionary into the UserDetails model
jane_user_model = UserDetails(**jane_user_details)

print(jane_user_model)

In [ ]:
users: list[UserDetails] = []
users.append(UserDetails(name="Alice Smith", phone_number="+1987654321", additional_details=AdditionalDetails(), address=[]))
users.append(UserDetails(name="Bob Johnson", phone_number="+1123456789", additional_details=AdditionalDetails(), address=[]))
users.append(UserDetails(name="Charlie Brown", phone_number="+1098765432", additional_details=AdditionalDetails(), address=[]))

# Printing BaseModel list
print(users)

# Printing each BaseModel in JSON format
for user in users:
    print(user.model_dump_json(indent=4))

### Using TypedDict

In [ ]:
from typing import TypedDict

class CompanyDetails(TypedDict):
    id: str
    company_name: str
    employee_count: int
    founded_year: int

In [ ]:
company_deatails = CompanyDetails(id=str(uuid.uuid4()), company_name="Tech Solutions")
company_deatails["employee_count"] = 150
company_deatails["founded_year"] = 2010

print(company_deatails)

### Inheritance

Inheritance allows a class (child class) to acquire properties and methods of another class (parent class). It supports hierarchical classification and promotes code reuse.

In [ ]:
class CarModel:
    def __init__(self, name: str, color: str):
        self.name = name
        self.color = color
        
    def get_model(self):
        return {
            "name": self.name,
            "color": self.color
        }
        
class CarEngine:
    def __init__(self, ename: str, is_petrol: bool):
        self.ename = ename
        self.is_petrol = is_petrol
        
    def get_engine(self):
        return {
            "name": self.ename,
            "is_petrol": self.is_petrol
        }
        
class Car(CarModel, CarEngine):
    def __init__(self, name: str, color: str, ename: str, is_petrol: bool):
        CarModel.__init__(self, name=name, color=color)
        CarEngine.__init__(self, ename=ename, is_petrol=is_petrol)
        
car_obj = Car(name="Altroz", color="White", ename="Revotron 1.2L Turbo", is_petrol=True)
print(car_obj.get_model())
print(car_obj.get_engine())

### Polymorphism

It allows methods with the same name to work differently. There are 2 concepts in Polymorphism - Method Overloading & Method Overriding

In [ ]:
# Method Overloading - Having multiple methods with the same name but different parameters.
# Note on Python: Python does not support traditional method overloading like Java or C++. 
# If you define two methods with the same name, the second one replaces the first. 
# However, we achieve similar behavior using default arguments or variable-length arguments (*args).

class Calculator:
    def add(self, *args):
        return sum(args)

calc = Calculator()
print(calc.add(5, 10))       # Two arguments
print(calc.add(5, 10, 15))   # Three arguments
print(calc.add(1, 2, 3, 4))  # Any number of arguments

In [ ]:
# Method Overriding - Providing a new implementation for a parent class method in the child class.

# We start with a base class and then a subclass that "overrides" the speak method.
class Animal:
    def speak(self):
        return "I am an animal."

class Dog(Animal):
    def speak(self):
        return "Woof!"

print(Dog().speak())